# 01. EDA & Data Overview v2

In [ ]:
import sys
import warnings
import os
from pathlib import Path

# Tìm thư mục gốc có chứa thư mục 'src' và thêm nó vào sys.path để có thể import các module từ src
current_path = Path.cwd().resolve()
PROJECT_ROOT = current_path
for parent in [current_path] + list(current_path.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import các thư viện
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.load_data import ABALONE_COLUMNS, load_abalone_data
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 15

## 1. Load Data

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'

# Gọi hàm load_abalone_data() từ src.data.load_data để tải dữ liệu và hiển thị 5 dòng đầu tiên của DataFrame
columns = ABALONE_COLUMNS.copy()
df = load_abalone_data(DATA_PATH, columns=columns)
display(df.head())


,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## 2. Tổng quan cấu trúc và kiểu dữ liệu (Shapes & Dtypes)

In [ ]:
# Tạo DataFrame để mô tả ý nghĩa của các cột và kiểu dữ liệu
data_dict = pd.DataFrame({
    'Mô tả': [
        'Giới tính: M (Đực), F (Cái), I (Con non)', 'Chiều dài vỏ lớn nhất (mm)',
        'Đường kính vỏ (mm)', 'Chiều cao (vỏ + thịt) (mm)', 'Khối lượng toàn bộ (g)',
        'Khối lượng thịt (g)', 'Khối lượng nội tạng (g)', 'Khối lượng vỏ khô (g)',
        'Số vòng tăng trưởng (Biến mục tiêu)'
    ]
}, index=df.columns)

data_dict['Kiểu dữ liệu'] = df.dtypes

print(f"Kích thước dữ liệu: {df.shape[0]} dòng x {df.shape[1]} cột")
display(data_dict)

Kích thước dữ liệu: 4177 dòng x 9 cột


,Mô tả,Kiểu dữ liệu
Sex,"Giới tính: M (Đực), F (Cái), I (Con non)",str
Length,Chiều dài vỏ lớn nhất (mm),float64
Diameter,Đường kính vỏ (mm),float64
Height,Chiều cao (vỏ + thịt) (mm),float64
WholeWeight,Khối lượng toàn bộ (g),float64
ShuckedWeight,Khối lượng thịt (g),float64
VisceraWeight,Khối lượng nội tạng (g),float64
ShellWeight,Khối lượng vỏ khô (g),float64
Rings,Số vòng tăng trưởng (Biến mục tiêu),int64


## 3. Kiểm tra tính toàn vẹn của dữ liệu

In [92]:
missing = df.isna().sum().to_frame('Số lượng thiếu')
missing['Tỷ lệ (%)'] = (missing['Số lượng thiếu'] / len(df) * 100).round(2)
duplicates = df.duplicated().sum()

print(f"Số dòng trùng lặp: {duplicates}")
if missing['Số lượng thiếu'].sum() == 0:
    print("Trạng thái: Dữ liệu sạch, không có giá trị thiếu.")
else:
    display(missing[missing['Số lượng thiếu'] > 0])

Số dòng trùng lặp: 0
Trạng thái: Dữ liệu sạch, không có giá trị thiếu.


## 4. Thống kê mô tả

In [ ]:
print("Thống kê mô tả các đặc trưng số:")
display(df.describe().T.style.background_gradient(cmap='Blues'))

# Kiểm tra nhanh các giá trị vật lý không hợp lệ, ví dụ: Height = 0
zero_height = (df['Height'] == 0).sum()
if zero_height > 0:
    print(f"Cảnh báo: Có {zero_height} dòng có Height = 0 (Lỗi vật lý cần xử lý).")

Thống kê mô tả các đặc trưng số:


,count,mean,std,min,25%,50%,75%,max
Length,4177.000000,0.523992,0.120093,0.075000,0.450000,0.545000,0.615000,0.815000
Diameter,4177.000000,0.407881,0.099240,0.055000,0.350000,0.425000,0.480000,0.650000
Height,4177.000000,0.139516,0.041827,0.000000,0.115000,0.140000,0.165000,1.130000
WholeWeight,4177.000000,0.828742,0.490389,0.002000,0.441500,0.799500,1.153000,2.825500
ShuckedWeight,4177.000000,0.359367,0.221963,0.001000,0.186000,0.336000,0.502000,1.488000
VisceraWeight,4177.000000,0.180594,0.109614,0.000500,0.093500,0.171000,0.253000,0.760000
ShellWeight,4177.000000,0.238831,0.139203,0.001500,0.130000,0.234000,0.329000,1.005000
Rings,4177.000000,9.933684,3.224169,1.000000,8.000000,9.000000,11.000000,29.000000


Cảnh báo: Có 2 dòng có Height = 0 (Lỗi vật lý cần xử lý).
